# Regressão ordinal: 8–10 variáveis e pseudo-R²
Resultados congelados. A execução abaixo não retreina nem seleciona com base no teste.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
p=Path.cwd()
display(Markdown((p/'RELATORIO_ITERACOES.md').read_text()))

# Iterações com 8, 9 e 10 variáveis

Execução: 28/10/2026. Base mock fornecida. Mesma política de referência da v1; aumentamos as variáveis de entrada, sem mudar os alvos para forçar importância das variáveis novas.

## Resultado por iteração

| n_variaveis | alpha | pseudo_r2_mcfadden_treino | pseudo_r2_preditivo_cv | acuracia_cv | log_loss_cv |
| --- | --- | --- | --- | --- | --- |
| 8 | 1.0 | 0.648767 | 0.645772 | 0.834788 | 0.378808 |
| 9 | 1.0 | 0.64885 | 0.644934 | 0.833109 | 0.379705 |
| 10 | 1.0 | 0.649022 | 0.644804 | 0.832774 | 0.379844 |

Foram executados 27 ajustes: três conjuntos de variáveis, três penalidades e três dobras separadas por empresa. A tabela mostra a melhor penalidade de cada conjunto; todos os detalhes estão em `cada_iteracao_e_dobra.csv` e `ranking_cv.csv`. A versão selecionada usa **8 variáveis**, com penalidade 1. A diferença entre as versões é pequena: não afirmamos superioridade estatística comprovada.

## O que significa R² aqui

Usamos **pseudo-R² de McFadden = 1 − LL_modelo/LL_nulo**, conforme [documentação statsmodels](https://www.statsmodels.org/stable/dev/generated/statsmodels.base.model.GenericLikelihoodModelResults.html). LL é a soma dos logaritmos das probabilidades da referência. A penalidade L2 não é incluída nessa soma. O modelo nulo usa apenas frequências dos planos no treino.

No treino, a tabela informa o índice de ajuste McFadden nos parâmetros penalizados. Na validação, usamos a mesma razão como **análogo preditivo**, com frequências aprendidas apenas no treino de cada dobra. Não é o pseudo-R² clássico ajustado na própria validação. O resultado agregado soma as log-verossimilhanças das previsões fora da amostra antes de calcular a razão. Um valor negativo seria possível se as previsões fossem piores que o modelo nulo.

**0,6458 não significa 64,58% da variância explicada nem 64,58% de acerto.** Selecionamos pela menor log-loss de validação, equivalente a maximizar essa razão quando o denominador é o mesmo. A regularização e as referências derivadas impedem interpretar este índice como prova de significância.

## Variáveis

| Iteração | Variáveis acrescentadas | Justificativa e limite |
|---|---|---|
| 8 | Funcionários; estoque; bens; faturamento; quantidade de estabelecimentos; atividade; tipo de unidade; área | Patrimônio, emprego, escala e contexto. Funcionários e faturamento referem-se à empresa, não ao endereço. Área tem muitos valores ausentes. |
| 9 | Presença do termo loja no endereço | Contexto exploratório, não confirmação de vitrine ou risco. |
| 10 | Presença do termo galeria no endereço | Contexto exploratório, não evidência de proteção ou exposição. |

São 8 variáveis originais no modelo escolhido, que viram **17 coeficientes** após categorias e indicador de ausência. Atividade tem vários pesos, relativos à categoria-base BELEZA; tipo de unidade usa FILIAL como base. Não existe um peso único de atividade sem especificar sua categoria. A regressão continua ordinal, com um único índice.

## Pesos e estabilidade

| termo | peso | ic_condicional_2_5 | ic_condicional_97_5 | sinal_estavel_condicional |
| --- | --- | --- | --- | --- |
| num__funcionarios_estimados_empresa | 2.890323 | 2.513148 | 3.355206 | True |
| num__estoque_medio_estimado | 1.302932 | 1.14481 | 1.472943 | True |
| num__valor_bens_ponderado_estabelecimento | 1.018984 | 0.782641 | 1.26607 | True |
| num__faturamento_mensal_empresa | 0.100012 | -0.399497 | 0.411498 | False |
| num__quantidade_estabelecimentos_empresa | -0.137338 | -0.29745 | -0.03159 | True |
| num__area_m2 | 0.059618 | -0.054542 | 0.169789 | False |
| num__missingindicator_area_m2 | 0.003361 | -0.107255 | 0.134345 | False |
| cat__grupo_atividade_CLINICA | 0.243811 | -0.178384 | 0.66629 | False |
| cat__grupo_atividade_ELETRONICOS | 0.20616 | -0.221763 | 0.694726 | False |
| cat__grupo_atividade_ESCOLA_CURSO | 0.201755 | -0.271505 | 0.570321 | False |
| cat__grupo_atividade_ESCRITORIO | 0.067674 | -0.294724 | 0.530529 | False |
| cat__grupo_atividade_FARMACIA | 0.584133 | 0.220909 | 1.013432 | True |
| cat__grupo_atividade_MERCADO | 0.294332 | -0.086985 | 0.655885 | False |
| cat__grupo_atividade_OFICINA | 0.036316 | -0.229433 | 0.443015 | False |
| cat__grupo_atividade_RESTAURANTE | -0.162121 | -0.641332 | 0.192504 | False |
| cat__grupo_atividade_VAREJO_MODA | 0.588416 | 0.237332 | 0.981181 | True |
| cat__tipo_unidade_MATRIZ | 0.227888 | -0.123102 | 0.52441 | False |

Pesos numéricos são para variáveis transformadas por log(1+x) e padronizadas. Indicadores de ausência também passam pela transformação do pipeline. Categorias são indicadores 0/1 com base omitida. A soma das contribuições produz z; sigmoid(z) produz o score. Fórmula e parâmetros completos em `formula_completa.json`.

100 reamostragens por empresa indicam estabilidade **condicional** à política, transformação e seleção. Faturamento, área e o indicador de ausência da área têm intervalos que cruzam zero. Logo, não foram confirmados como efeitos estáveis nesta referência. Funcionários, estoque e bens mantêm sinais positivos estáveis; quantidade de estabelecimentos tem sinal negativo condicional, sem interpretação causal. Alguns contrastes de atividade também têm estabilidade, mas não são recomendação de seguro por setor.

Os critérios de referência continuam baseados em três variáveis. Acrescentar cinco preditores não cria informação de adequação que não existe no alvo. Não é defensável dizer que as oito variáveis são estatisticamente significativas.

## Novas faixas, próprias da versão 2

- Essencial: score abaixo de 0.284127232.
- Prático: de 0.284127232 até abaixo de 0.989682875.
- Total: de 0.989682875 em diante.

Score = sigmoid(X beta); faixas = sigmoid dos dois limiares ajustados. Decisão pela mediana ordinal, não necessariamente pela maior probabilidade. Não arredondar antes de enquadrar. Não misturar scores da v2 com faixas da v1.

## Reavaliação do teste comum

Acertos: **114/144 = 79.17%**, ante 116/144 = 80,56% na v1. Pseudo-R² preditivo: 0.631592. Log-loss: 0.391467. Assim, **o requisito de pelo menos oito variáveis foi atendido, mas não melhorou o resultado observado**. Não ajustamos a seleção com base nessa reavaliação.

O teste já havia sido consultado na versão anterior; esta é uma comparação na mesma base, não uma nova avaliação independente intocada. A concordância é com critérios construídos, não com planos ideais observados. A limitação ordinal frente ao comparador multinomial encontrada na v1 não foi resolvida por acrescentar variáveis.

## Entregáveis e verificações

`perfis_rag.jsonl` contém 144 perfis da versão 2. `modelo_ordinal.pkl` contém a transformação e parâmetros. `inferir.py` aplica novos lotes sem treinar. `testes.json` registra 11 verificações aprovadas e `teste_inferencia.json` confirma reprodução dos 144 scores e planos após carregar o modelo. O notebook apresenta a execução congelada.


In [2]:
display(pd.read_csv(p/'resultados/resumo_8_9_10.csv'))
display(pd.read_csv(p/'resultados/cada_iteracao_e_dobra.csv'))

,iteracao,n_variaveis,alpha,log_loss_cv,pseudo_r2_preditivo_cv,acuracia_cv,macro_f1_cv,pseudo_r2_mcfadden_treino,acuracia_treino,n_coeficientes_apos_codificacao
0,8_variaveis,8,1.0,0.378808,0.645772,0.834788,0.829891,0.648767,0.837475,17
1,9_variaveis,9,1.0,0.379705,0.644934,0.833109,0.828294,0.648850,0.837811,18
2,10_variaveis,10,1.0,0.379844,0.644804,0.832774,0.827911,0.649022,0.836803,19


,iteracao,n_variaveis,alpha,dobra,treino_log_loss,treino_log_loss_nulo,treino_indice_1_menos_LL_LL0,treino_acuracia,treino_macro_f1,validacao_log_loss,validacao_log_loss_nulo,validacao_indice_1_menos_LL_LL0,validacao_acuracia,validacao_macro_f1
0,8_variaveis,8,0.1,1,0.376730,1.069515,0.647756,0.836272,0.833048,0.372754,1.062348,0.649122,0.833837,0.830267
1,8_variaveis,8,0.1,2,0.365480,1.069155,0.658160,0.840806,0.835796,0.378019,1.068519,0.646221,0.846928,0.841488
2,8_variaveis,8,0.1,3,0.367366,1.067588,0.655891,0.839376,0.833537,0.387416,1.077318,0.640389,0.824597,0.821154
3,8_variaveis,8,1.0,1,0.376949,1.069515,0.647552,0.838287,0.834686,0.371899,1.062348,0.649927,0.835851,0.831438
4,8_variaveis,8,1.0,2,0.365841,1.069155,0.657823,0.840806,0.835908,0.376103,1.068519,0.648015,0.843907,0.837917
5,8_variaveis,8,1.0,3,0.367490,1.067588,0.655776,0.840886,0.834729,0.388432,1.077318,0.639445,0.824597,0.820260
6,8_variaveis,8,10.0,1,0.382470,1.069515,0.642390,0.836776,0.832241,0.371979,1.062348,0.649852,0.837865,0.832949
7,8_variaveis,8,10.0,2,0.372187,1.069155,0.651887,0.832746,0.826406,0.374917,1.068519,0.649125,0.847936,0.841222
8,8_variaveis,8,10.0,3,0.370887,1.067588,0.652594,0.838369,0.831510,0.393844,1.077318,0.634422,0.822581,0.819051
9,9_variaveis,9,0.1,1,0.376110,1.069515,0.648336,0.840302,0.837155,0.374547,1.062348,0.647435,0.836858,0.832521


In [3]:
display(pd.read_csv(p/'resultados/pesos.csv'))
display(json.loads((p/'resultados/formula_completa.json').read_text()))

,termo,peso,ic_condicional_2_5,ic_condicional_97_5,sinal_estavel_condicional
0,num__funcionarios_estimados_empresa,2.890323,2.513148,3.355206,True
1,num__estoque_medio_estimado,1.302932,1.144810,1.472943,True
2,num__valor_bens_ponderado_estabelecimento,1.018984,0.782641,1.266070,True
3,num__faturamento_mensal_empresa,0.100012,-0.399497,0.411498,False
4,num__quantidade_estabelecimentos_empresa,-0.137338,-0.297450,-0.031590,True
5,num__area_m2,0.059618,-0.054542,0.169789,False
6,num__missingindicator_area_m2,0.003361,-0.107255,0.134345,False
7,cat__grupo_atividade_CLINICA,0.243811,-0.178384,0.666290,False
8,cat__grupo_atividade_ELETRONICOS,0.206160,-0.221763,0.694726,False
9,cat__grupo_atividade_ESCOLA_CURSO,0.201755,-0.271505,0.570321,False


{'versao': 'ordinal_mock_v2',
 'formula': 'z=sum(beta_j*x_j); score=sigmoid(z); x é a transformação descrita abaixo',
 'colunas_numericas': ['funcionarios_estimados_empresa',
  'estoque_medio_estimado',
  'valor_bens_ponderado_estabelecimento',
  'faturamento_mensal_empresa',
  'quantidade_estabelecimentos_empresa',
  'area_m2'],
 'medianas_imputacao': [6.0, 25700.0, 131575.0, 191750.0, 1.0, 400.0],
 'indices_indicadores_ausencia': [5],
 'transformacao_numerica': 'Imputar; anexar indicadores de ausência; log1p de todas estas colunas; (valor-media)/desvio',
 'medias_apos_log1p': [2.0901357727829817,
  10.102767825413773,
  11.774504398967503,
  12.200984922915827,
  0.8233099217793559,
  5.9186553349555,
  0.4278054123133577],
 'desvios_apos_log1p': [0.9046646004797476,
  1.7594636346908492,
  1.3528137825976112,
  1.1035235118284972,
  0.18929899824994345,
  0.44410455153831396,
  0.33691934430763515],
 'colunas_categoricas': ['grupo_atividade', 'tipo_unidade'],
 'categorias': [['BELEZ

In [4]:
display(json.loads((p/'resultados/testes.json').read_text()))
display(json.loads((p/'resultados/perfis_rag.jsonl').read_text().splitlines()[0]))

{'8_a_10_variaveis': True,
 '3_iteracoes_9_configuracoes_27_ajustes': True,
 'politica_v1_preservada': True,
 'p_somam_1': True,
 'p_validas': True,
 'score_valido': True,
 'cortes_ordenados': True,
 'mediana_confere': True,
 'contribuicoes_conferem': True,
 'modelo_congelado': True,
 'bootstrap_90_ou_mais': True}

{'versao': 'ordinal_mock_v2',
 'status': 'EXPERIMENTAL_BASE_MOCK',
 'empresa_id': 'CASO_00001',
 'imovel_id': 'IMO_CASO_00001_01',
 'score': 0.9980057003502676,
 'plano_sugerido': 'TOTAL',
 'cortes': [0.2841272318707447, 0.9896828745948337],
 'probabilidades_referencia': {'ESSENCIAL': 0.0007924832681200758,
  'PRATICO': 0.16006164125126526,
  'TOTAL': 0.8391458754806147},
 'contribuicoes': [{'termo': 'num__funcionarios_estimados_empresa',
   'contribuicao_z': 4.188726038110717},
  {'termo': 'num__estoque_medio_estimado',
   'contribuicao_z': 0.5152647054635189},
  {'termo': 'num__valor_bens_ponderado_estabelecimento',
   'contribuicao_z': 1.1678692730526576},
  {'termo': 'num__faturamento_mensal_empresa',
   'contribuicao_z': 0.1706474368203962},
  {'termo': 'num__quantidade_estabelecimentos_empresa',
   'contribuicao_z': 0.09443442398701897},
  {'termo': 'num__area_m2', 'contribuicao_z': 0.010109371010759222},
  {'termo': 'num__missingindicator_area_m2',
   'contribuicao_z': 0.0026471